<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/mixtral_to_gguf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://huggingface.co/frankmorales2020/Mixtral-8x22B-topo-cbp

## setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!rm -rf /content/drive/MyDrive/topogpt/mixtral_8x22b_q4_k_m.gguf

In [ ]:
!ls -ltha /content/drive/MyDrive/topogpt/

total 219G
drwx------ 48 root root 4.0K Nov  4 09:43 ..
-rw-------  1 root root  80G Nov  4 06:49 mixtral_8x22b_q4_k_m.gguf
drwx------  2 root root 4.0K Nov  4 06:36 tmpdev
-rw-------  1 root root  29G Nov  4 02:30 mixtral_8x22b_q8_0-00004-of-00004.gguf
-rw-------  1 root root  38G Nov  4 00:40 mixtral_8x22b_q8_0-00003-of-00004.gguf
-rw-------  1 root root  38G Nov  3 21:46 mixtral_8x22b_q8_0-00002-of-00004.gguf
-rw-------  1 root root  37G Nov  3 18:55 mixtral_8x22b_q8_0-00001-of-00004.gguf
drwx------  2 root root 4.0K Nov  3 04:26 mixtral_8x22b_bf16_hf
drwx------  2 root root 4.0K Nov  2 11:04 llama.cpp
drwx------  2 root root 4.0K Nov  1 17:34 tmp
drwx------  2 root root 4.0K Nov  1 12:22 base_model_70b
drwx------  2 root root 4.0K Nov  1 12:22 checkpoints
drwx------  2 root root 4.0K Oct 26 15:10 gguf
drwx------  2 root root 4.0K Oct 19 19:52 hf_cache
drwx------  2 root root 4.0K Oct 15 01:11 mixtral-8x22b-topocbp-nolora
drwx------ 11 root root 4.0K Oct  9 10:40 .


In [ ]:
!ls -ltha /content/drive/MyDrive/topogpt/llama.cpp/ ; ls /content/drive/MyDrive/topogpt/llama.cpp/build/bin/

In [ ]:
import os
import sys
import time
import subprocess
from pathlib import Path
from google.colab import drive, userdata
from huggingface_hub import HfApi, hf_hub_download

# =====================================================================
# 1. MOUNT DRIVE & SETUP DIRECTORY REFERENCES
# =====================================================================
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

DRIVE_OUT = "/content/drive/MyDrive/topogpt"
LLAMA_DIR = os.path.join(DRIVE_OUT, "llama.cpp")
BUILD_DIR = os.path.join(LLAMA_DIR, "build")
QUANT_BIN = os.path.join(BUILD_DIR, "bin/llama-quantize")
CONVERT_PY = os.path.join(LLAMA_DIR, "convert_hf_to_gguf.py")

# Intermediate files on fast local NVMe (/content)
LOCAL_HF_DIR = "/content/mixtral_8x22b_hf"
TEMP_BF16 = "/content/mixtral_8x22b_bf16.gguf"

# Final quantized artifact persisted directly to Google Drive
FINAL_Q4 = os.path.join(DRIVE_OUT, "Mixtral-8x22B-topo-cbp-q4_k_m.gguf")

REPO_ID = "frankmorales2020/Mixtral-8x22B-topo-cbp"
HF_TOKEN = userdata.get("HF_TOKEN")

os.makedirs(LOCAL_HF_DIR, exist_ok=True)
os.makedirs(DRIVE_OUT, exist_ok=True)

assert os.path.exists(CONVERT_PY), f"Converter script missing: {CONVERT_PY}"
assert os.path.exists(QUANT_BIN), f"Quantizer binary missing: {QUANT_BIN}"
os.chmod(QUANT_BIN, 0o755)

# =====================================================================
# 2. CONFIGURE DYNAMIC LINKER ENVIRONMENT
# =====================================================================
env = os.environ.copy()
lib_paths = [
    os.path.join(BUILD_DIR, "bin"),
    os.path.join(BUILD_DIR, "src"),
    os.path.join(BUILD_DIR, "ggml/src"),
    BUILD_DIR,
    env.get("LD_LIBRARY_PATH", "")
]
env["LD_LIBRARY_PATH"] = ":".join(p for p in lib_paths if p)

print(f"Using llama.cpp base:  {LLAMA_DIR}")
print(f"Using quantize binary: {QUANT_BIN}")
print(f"LD_LIBRARY_PATH set to: {env['LD_LIBRARY_PATH']}")

# =====================================================================
# 3. DOWNLOAD SHARDS (RATE-LIMIT & 429 RESILIENT)
# =====================================================================
print(f"\nFetching repository file manifest for {REPO_ID}...")
api = HfApi(token=HF_TOKEN)
repo_files = api.list_repo_files(repo_id=REPO_ID)

# Target configuration, tokenizer, and safetensors shards only
filtered_files = [
    f for f in repo_files
    if not any(f.endswith(ext) for ext in [".msgpack", ".h5", ".ot", ".gitattributes"])
]

print(f"Total files to synchronize: {len(filtered_files)}")

for idx, file_name in enumerate(filtered_files, 1):
    dest_file = os.path.join(LOCAL_HF_DIR, file_name)

    # Check if shard already downloaded completely
    if os.path.exists(dest_file):
        print(f"[{idx}/{len(filtered_files)}] Already cached: {file_name}")
        continue

    print(f"[{idx}/{len(filtered_files)}] Downloading: {file_name}...")
    success = False
    attempts = 0
    max_retries = 5

    while not success and attempts < max_retries:
        try:
            hf_hub_download(
                repo_id=REPO_ID,
                filename=file_name,
                local_dir=LOCAL_HF_DIR,
                token=HF_TOKEN,
                resume_download=True,
            )
            success = True
            time.sleep(1.0)  # Inter-shard cooldown to prevent CDN burst limits
        except Exception as e:
            attempts += 1
            wait_time = 5 * (2 ** attempts)
            print(f"  Warning: Encountered {e}. Retrying in {wait_time}s (Attempt {attempts}/{max_retries})...")
            time.sleep(wait_time)

    if not success:
        raise RuntimeError(f"Failed to fetch {file_name} after {max_retries} attempts.")

print(f"\nAll shards successfully retrieved to {LOCAL_HF_DIR}.")

In [ ]:
!cp -pr /content/drive/MyDrive/topogpt/mixtral_8x22b_hf /content/

In [ ]:
!ls -ltha /content/mixtral_8x22b_hf/

In [ ]:
!pip install -q bitsandbytes>=0.46.1

In [ ]:
import os
import gc
import sys
import json
import shutil
import subprocess
import torch
from pathlib import Path
from google.colab import drive
from safetensors import safe_open
from safetensors.torch import save_file
import bitsandbytes.functional as bnb_F

# =====================================================================
# 1. PATHS & DYNAMIC LINKER ENVIRONMENT
# =====================================================================
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

DRIVE_OUT = "/content/drive/MyDrive/topogpt"
LLAMA_DIR = os.path.join(DRIVE_OUT, "llama.cpp")
BUILD_DIR = os.path.join(LLAMA_DIR, "build")
QUANT_BIN = os.path.join(BUILD_DIR, "bin/llama-quantize")
CONVERT_PY = os.path.join(LLAMA_DIR, "convert_hf_to_gguf.py")

SRC_HF_DIR = "/content/drive/MyDrive/topogpt/mixtral_8x22b_hf"
DEQUANT_HF_DIR = "/content/drive/MyDrive/topogpt/mixtral_8x22b_bf16_hf"
OUTPUT_BF16_GGUF = "/content/drive/MyDrive/topogpt/mixtral_8x22b_bf16.gguf"
FINAL_Q4_GGUF = os.path.join(DRIVE_OUT, "Mixtral-8x22B-topo-cbp-q4_k_m.gguf")

# Clean previous failed conversion cache if present
if os.path.exists(DEQUANT_HF_DIR):
    shutil.rmtree(DEQUANT_HF_DIR)
os.makedirs(DEQUANT_HF_DIR, exist_ok=True)
os.makedirs(DRIVE_OUT, exist_ok=True)

if os.path.exists(QUANT_BIN):
    os.chmod(QUANT_BIN, 0o755)

env = os.environ.copy()
lib_paths = [
    os.path.join(BUILD_DIR, "bin"),
    os.path.join(BUILD_DIR, "src"),
    os.path.join(BUILD_DIR, "ggml/src"),
    BUILD_DIR,
    env.get("LD_LIBRARY_PATH", "")
]
env["LD_LIBRARY_PATH"] = ":".join(p for p in lib_paths if p)

# =====================================================================
# 2. PURIFIED SHARD-BY-SHARD DEQUANTIZATION
# =====================================================================
print("--- [1/3] Dequantizing BitsAndBytes shards to pure BF16 safetensors ---")

with open(os.path.join(SRC_HF_DIR, "config.json"), "r") as f:
    config = json.load(f)

# Strip bitsandbytes configuration
config.pop("quantization_config", None)
config["torch_dtype"] = "bfloat16"

with open(os.path.join(DEQUANT_HF_DIR, "config.json"), "w") as f:
    json.dump(config, f, indent=2)

for fname in os.listdir(SRC_HF_DIR):
    if fname.endswith(".json") and fname not in ("config.json", "model.safetensors.index.json"):
        shutil.copy(os.path.join(SRC_HF_DIR, fname), os.path.join(DEQUANT_HF_DIR, fname))

safetensors_files = sorted(Path(SRC_HF_DIR).glob("*.safetensors"))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Processing {len(safetensors_files)} shards using device: {device}...")

for sf_path in safetensors_files:
    print(f"  Unpacking {sf_path.name}...")
    tensors = {}
    with safe_open(sf_path, framework="pt", device="cpu") as f:
        all_keys = list(f.keys())

        # Real model weights only end with .weight or .bias
        true_keys = [k for k in all_keys if k.endswith(".weight") or k.endswith(".bias")]

        for k in true_keys:
            w = f.get_tensor(k)

            # Check if this weight is 4-bit quantized
            bnb_prefix = f"{k}."
            state_keys = {sk: f.get_tensor(sk) for sk in all_keys if sk.startswith(bnb_prefix)}

            if w.dtype == torch.uint8 and len(state_keys) > 0:
                # Reconstruct QuantState payload for bitsandbytes
                qs_dict = {}
                for sk, sv in state_keys.items():
                    rel_key = sk[len(bnb_prefix):]
                    qs_dict[rel_key] = sv.to(device)

                try:
                    quant_state = bnb_F.QuantState.from_dict(qs_dict, device=device)
                    dequant_w = bnb_F.dequantize_4bit(w.to(device), quant_state=quant_state)
                    tensors[k] = dequant_w.to(dtype=torch.bfloat16, device="cpu")
                    del dequant_w, quant_state
                except Exception:
                    # Fallback for flat unnested quant states
                    absmax_key = f"{k}.absmax"
                    absmax = f.get_tensor(absmax_key).to(device) if absmax_key in all_keys else None
                    dequant_w = bnb_F.dequantize_4bit(w.to(device), absmax=absmax, quant_type="nf4")
                    tensors[k] = dequant_w.to(dtype=torch.bfloat16, device="cpu")
                    del dequant_w
            else:
                tensors[k] = w.to(torch.bfloat16)

    out_shard = os.path.join(DEQUANT_HF_DIR, sf_path.name)
    save_file(tensors, out_shard)
    del tensors
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Sanitize model.safetensors.index.json to include only pure weight keys
src_index_file = os.path.join(SRC_HF_DIR, "model.safetensors.index.json")
if os.path.exists(src_index_file):
    with open(src_index_file, "r") as f:
        idx = json.load(f)
    sanitized_map = {
        k: v for k, v in idx.get("weight_map", {}).items()
        if k.endswith(".weight") or k.endswith(".bias")
    }
    idx["weight_map"] = sanitized_map
    with open(os.path.join(DEQUANT_HF_DIR, "model.safetensors.index.json"), "w") as f:
        json.dump(idx, f, indent=2)

#print("\nPure BF16 safetensors generated with no auxiliary metadata contamination.\n")

!rm -rf /content/drive/MyDrive/topogpt/mixtral_8x22b_hf

print("\nPure BF16 safetensors generated with no auxiliary metadata contamination.\n")


# =====================================================================
# 3. GGUF CONVERSION (CONVERT_HF_TO_GGUF.PY)
# =====================================================================
print(f"--- [2/3] Converting {DEQUANT_HF_DIR} -> {OUTPUT_BF16_GGUF} ---")
convert_cmd = [
    sys.executable,
    CONVERT_PY,
    DEQUANT_HF_DIR,
    "--outtype", "bf16",
    "--outfile", OUTPUT_BF16_GGUF,
    "--verbose"
]

proc = subprocess.Popen(
    convert_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env
)

for line in proc.stdout:
    print(line, end="")
    sys.stdout.flush()

proc.wait()

if proc.returncode != 0:
    raise RuntimeError(f"GGUF conversion failed with exit code {proc.returncode}")

print(f"\nBase GGUF created: {OUTPUT_BF16_GGUF} ({os.path.getsize(OUTPUT_BF16_GGUF) / 1024**3:.2f} GB)\n")

# Free intermediate dequantized shards
shutil.rmtree(DEQUANT_HF_DIR, ignore_errors=True)

# =====================================================================
# 4. QUANTIZE TO Q4_K_M VIA LLAMA-QUANTIZE
# =====================================================================
if os.path.exists(QUANT_BIN):
    print(f"--- [3/3] Quantizing to {FINAL_Q4_GGUF} (Q4_K_M) ---")
    quant_cmd = [QUANT_BIN, OUTPUT_BF16_GGUF, FINAL_Q4_GGUF, "Q4_K_M"]
    subprocess.run(quant_cmd, check=True, env=env)

    if os.path.exists(OUTPUT_BF16_GGUF):
        os.remove(OUTPUT_BF16_GGUF)
        print(f"Purged intermediate BF16 GGUF: {OUTPUT_BF16_GGUF}")

    print(f"\nPipeline finished. Certified GGUF: {FINAL_Q4_GGUF}")
    print(f"Final size: {os.path.getsize(FINAL_Q4_GGUF) / 1024**3:.2f} GB")

In [ ]:
!!df -h /content /content/drive/MyDrive

['Filesystem      Size  Used Avail Use% Mounted on',
 'overlay         236G  234G  2.1G 100% /',
 'drive           236G  234G  2.0G 100% /content/drive']

In [ ]:
!du -sh /content/drive/MyDrive/topogpt/* /content/* 2>/dev/null | sort -hr | head -n 15

^C


In [ ]:
!du -sh /root/.config/Google/DriveFS/*

65M	/root/.config/Google/DriveFS/107621623640896046014
4.0K	/root/.config/Google/DriveFS/cello_assert_history
28K	/root/.config/Google/DriveFS/experiments.db
4.0K	/root/.config/Google/DriveFS/first-run-info
0	/root/.config/Google/DriveFS/flags_fetched
4.0K	/root/.config/Google/DriveFS/global_feature_config
45M	/root/.config/Google/DriveFS/Logs
230G	/root/.config/Google/DriveFS/lost_and_found
12K	/root/.config/Google/DriveFS/metrics_store_sqlite.db
32K	/root/.config/Google/DriveFS/metrics_store_sqlite.db-shm
80K	/root/.config/Google/DriveFS/metrics_store_sqlite.db-wal
0	/root/.config/Google/DriveFS/onboarding_data
4.0K	/root/.config/Google/DriveFS/pid.txt
4.0K	/root/.config/Google/DriveFS/preferences.json
36K	/root/.config/Google/DriveFS/root_preference_sqlite.db
32K	/root/.config/Google/DriveFS/root_preference_sqlite.db-shm
0	/root/.config/Google/DriveFS/root_preference_sqlite.db-wal
8.0K	/root/.config/Google/DriveFS/server_feature_flags
4.0K	/root/.config/Google/DriveFS/ShellIpcPath
4

In [ ]:
!rm -rf /root/.config/Google/DriveFS/lost_and_found/*

In [ ]:
!df -h / /content/drive

df: /content/drive: No such file or directory
Filesystem      Size  Used Avail Use% Mounted on
overlay         236G   15G  221G   7% /


In [ ]:
!rm -f /content/drive/MyDrive/topogpt/mixtral_8x22b*.gguf

In [ ]:
!ls -ltha /content/drive/MyDrive/topogpt/mixtral*

/content/drive/MyDrive/topogpt/mixtral_8x22b_bf16_hf:
total 262G
drwx------ 10 root root 4.0K Nov  3 11:06 ..
-rw-------  1 root root 159K Nov  3 05:14 model.safetensors.index.json
-rw-------  1 root root  13G Nov  3 05:14 model-00015-of-00015.safetensors
-rw-------  1 root root  18G Nov  3 05:12 model-00014-of-00015.safetensors
-rw-------  1 root root  18G Nov  3 05:09 model-00013-of-00015.safetensors
-rw-------  1 root root  18G Nov  3 05:06 model-00012-of-00015.safetensors
-rw-------  1 root root  18G Nov  3 05:02 model-00011-of-00015.safetensors
-rw-------  1 root root  18G Nov  3 04:58 model-00010-of-00015.safetensors
-rw-------  1 root root  18G Nov  3 04:54 model-00009-of-00015.safetensors
-rw-------  1 root root  18G Nov  3 04:51 model-00008-of-00015.safetensors
-rw-------  1 root root  18G Nov  3 04:47 model-00007-of-00015.safetensors
-rw-------  1 root root  18G Nov  3 04:44 model-00006-of-00015.safetensors
-rw-------  1 root root  18G Nov  3 04:41 model-00005-of-00015.safete

## convert_hf_to_gguf

In [ ]:
!python /content/drive/MyDrive/topogpt/llama.cpp/convert_hf_to_gguf.py \
  /content/drive/MyDrive/topogpt/mixtral_8x22b_bf16_hf \
  --outtype q8_0 \
  --split-max-size 40G \
  --outfile /content/drive/MyDrive/topogpt/mixtral_8x22b_q8_0.gguf \
  --verbose

INFO:hf-to-gguf:Loading model: mixtral_8x22b_bf16_hf
INFO:hf-to-gguf:Model architecture: MixtralForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00015.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00015.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00003-of-00015.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00004-of-00015.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00005-of-00015.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00006-of-00015.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00007-of-00015.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00008-of-00015.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00009-of-00015.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00010-of-00015.safetensors'
INFO:hf-to-gguf:gguf: indexing model pa

## llama-quantize

In [ ]:
!chmod a+x /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-quantize

In [ ]:
!ls -ltha /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-quantize

-rwx------ 1 root root 18K Nov  2 10:56 /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-quantize


In [ ]:
!/content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-quantize \
  /content/drive/MyDrive/topogpt/mixtral_8x22b_q8_0-00001-of-00004.gguf \
  /content/drive/MyDrive/topogpt/mixtral_8x22b_q4_k_m.gguf \
  Q4_K_M

/content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-quantize: error while loading shared libraries: libllama-quantize-impl.so: cannot open shared object file: No such file or directory


In [ ]:
!LD_LIBRARY_PATH="/content/drive/MyDrive/topogpt/llama.cpp/build/bin:/content/drive/MyDrive/topogpt/llama.cpp/build/src:/content/drive/MyDrive/topogpt/llama.cpp/build/common:$LD_LIBRARY_PATH" \
/content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-quantize \
  /content/drive/MyDrive/topogpt/mixtral_8x22b_q8_0-00001-of-00004.gguf \
  /content/drive/MyDrive/topogpt/mixtral_8x22b_q4_k_m.gguf \
  Q4_K_M

In [ ]:
import os
import subprocess
import sys

# Step 1: Compile a fresh llama-quantize binary in /tmp matching this VM's CPU architecture
source_dir = "/content/drive/MyDrive/topogpt/llama.cpp"
tmp_build_dir = "/content/drive/MyDrive/topogpt/tmpdev/build_quant"
quantize_bin = os.path.join(tmp_build_dir, "bin/llama-quantize")

if not os.path.isfile(quantize_bin):
  print("Compiling llama-quantize for this specific VM host CPU...")
  cmake_configure = [
      "cmake",
      "-B",
      tmp_build_dir,
      "-S",
      source_dir,
      "-DCMAKE_BUILD_TYPE=Release",
      "-DGGML_NATIVE=ON",
      "-DGGML_AVX512=OFF",
      "-DGGML_CUDA=OFF",
  ]
  subprocess.run(cmake_configure, check=True)

  cpu_count = str(os.cpu_count() or 4)
  cmake_build = [
      "cmake",
      "--build",
      tmp_build_dir,
      "--config",
      "Release",
      "-j",
      cpu_count,
      "--target",
      "llama-quantize",
  ]
  subprocess.run(cmake_build, check=True)
  print("Compilation finished.\n")

# Step 2: Set LD_LIBRARY_PATH pointing to the compiled libraries in /tmp
local_bin_dir = os.path.join(tmp_build_dir, "bin")
os.environ["LD_LIBRARY_PATH"] = (
    f"{local_bin_dir}:{os.environ.get('LD_LIBRARY_PATH', '')}"
)

# Step 3: Run quantization across the 4 Q8_0 shards
input_shard = (
    "/content/drive/MyDrive/topogpt/mixtral_8x22b_q8_0-00001-of-00004.gguf"
)
output_gguf = "/content/drive/MyDrive/topogpt/mixtral_8x22b_q4_k_m.gguf"

cmd = [quantize_bin, input_shard, output_gguf, "Q4_K_M"]

print(f"Executing: {' '.join(cmd)}\n")

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
  sys.stdout.write(line)
  sys.stdout.flush()

process.wait()

if process.returncode != 0:
  raise RuntimeError(f"Quantization failed with exit code {process.returncode}")

print("\nModel successfully quantized to Q4_K_M!")

In [ ]:
!ls -ltha /content/drive/MyDrive/topogpt/tmpdev/build_quant/bin

total 15M
-rw-------  1 root root  88K Nov  4 06:38 libllama-quantize-impl.so
-rw-------  1 root root  18K Nov  4 06:38 llama-quantize
lrw-------  1 root root   20 Nov  4 06:38 libllama-common.so -> libllama-common.so.0
lrw-------  1 root root   24 Nov  4 06:38 libllama-common.so.0 -> libllama-common.so.0.8.0
-rw-------  1 root root 6.4M Nov  4 06:38 libllama-common.so.0.8.0
lrw-------  1 root root   13 Nov  4 06:37 libllama.so -> libllama.so.0
lrw-------  1 root root   17 Nov  4 06:37 libllama.so.0 -> libllama.so.0.8.0
-rw-------  1 root root 5.3M Nov  4 06:37 libllama.so.0.8.0
lrw-------  1 root root   12 Nov  4 06:37 libggml.so -> libggml.so.0
lrw-------  1 root root   17 Nov  4 06:37 libggml.so.0 -> libggml.so.0.28.0
-rw-------  1 root root  60K Nov  4 06:37 libggml.so.0.28.0
lrw-------  1 root root   16 Nov  4 06:37 libggml-cpu.so -> libggml-cpu.so.0
lrw-------  1 root root   21 Nov  4 06:37 libggml-cpu.so.0 -> libggml-cpu.so.0.28.0
-rw-------  1 root root 1.6M Nov  4 06:37 libggm

## llama-quantize-final

In [ ]:
import os
import subprocess
import sys
import time

# =====================================================================
# PATH CONFIGURATION
# =====================================================================
QUANTIZE_BIN = "/content/drive/MyDrive/topogpt/tmpdev/build_quant/bin/llama-quantize"
INPUT_SHARD = (
    "/content/drive/MyDrive/topogpt/mixtral_8x22b_q8_0-00001-of-00004.gguf"
)
OUTPUT_GGUF = "/content/drive/MyDrive/topogpt/mixtral_8x22b_q4_k_m.gguf"

# =====================================================================
# 1. ENVIRONMENT VALIDATION & PERMISSIONS
# =====================================================================
if not os.path.exists(QUANTIZE_BIN):
  raise FileNotFoundError(f"Quantize binary not found at: {QUANTIZE_BIN}")

if not os.path.exists(INPUT_SHARD):
  raise FileNotFoundError(f"Input shard manifest not found at: {INPUT_SHARD}")

os.chmod(QUANTIZE_BIN, 0o755)

print(f"[*] Quantization Binary : {QUANTIZE_BIN}")
print(f"[*] Input Shard 1       : {INPUT_SHARD}")
print(f"[*] Target Destination  : {OUTPUT_GGUF}")

# Configure runtime library path for llama.cpp dependencies
env = os.environ.copy()
build_dir = os.path.dirname(os.path.dirname(QUANTIZE_BIN))
lib_paths = [
    os.path.join(build_dir, "bin"),
    os.path.join(build_dir, "src"),
    os.path.join(build_dir, "ggml/src"),
    build_dir,
    env.get("LD_LIBRARY_PATH", ""),
]
env["LD_LIBRARY_PATH"] = ":".join(p for p in lib_paths if p)

# =====================================================================
# 2. RUNTIME PIPELINE EXECUTION
# =====================================================================
cmd = [QUANTIZE_BIN, "--allow-requantize", INPUT_SHARD, OUTPUT_GGUF, "Q4_K_M"]

print(f"\nExecuting: {' '.join(cmd)}\n" + "=" * 70)
start_time = time.time()

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

for line in process.stdout:
  sys.stdout.write(line)
  sys.stdout.flush()

process.wait()
elapsed = time.time() - start_time

if process.returncode != 0:
  raise RuntimeError(f"Quantization failed with exit code {process.returncode}")

print("=" * 70)
print(f"[*] Quantization exited cleanly in {elapsed / 60:.2f} minutes.")

# =====================================================================
# 3. FUSE CACHE FLUSH & INODE VERIFICATION
# =====================================================================
print("[*] Flushing OS dirty buffers to Google Drive FUSE...")
os.sync()
time.sleep(2)

if os.path.exists(OUTPUT_GGUF):
  file_bytes = os.path.getsize(OUTPUT_GGUF)
  size_gb = file_bytes / (1024**3)
  print(f"[+] Output file verified on filesystem:")
  print(f"    Raw bytes : {file_bytes:,} bytes")
  print(f"    Size (GB) : {size_gb:.2f} GB")

  if file_bytes == 0:
    print(
        "\n[!] Warning: Kernel still reports 0 bytes due to FUSE caching lag."
    )
    print("    Running explicit low-level stat call:")
    subprocess.run(["stat", OUTPUT_GGUF])
  else:
    print("\n[+] Verification Complete: Ready for zero-drift validation.")
else:
  print(f"[!] Error: Output file not visible at: {OUTPUT_GGUF}")

[*] Quantization Binary : /content/drive/MyDrive/topogpt/tmpdev/build_quant/bin/llama-quantize
[*] Input Shard 1       : /content/drive/MyDrive/topogpt/mixtral_8x22b_q8_0-00001-of-00004.gguf
[*] Target Destination  : /content/drive/MyDrive/topogpt/mixtral_8x22b_q4_k_m.gguf

Executing: /content/drive/MyDrive/topogpt/tmpdev/build_quant/bin/llama-quantize --allow-requantize /content/drive/MyDrive/topogpt/mixtral_8x22b_q8_0-00001-of-00004.gguf /content/drive/MyDrive/topogpt/mixtral_8x22b_q4_k_m.gguf Q4_K_M
version: 0.8.0-dev (build 1, commit 5969566)
built with GNU 13.3.0 for Linux x86_64
llama_quantize: quantizing '/content/drive/MyDrive/topogpt/mixtral_8x22b_q8_0-00001-of-00004.gguf' to '/content/drive/MyDrive/topogpt/mixtral_8x22b_q4_k_m.gguf' as Q4_K_M
llama_model_loader: additional 3 GGUFs metadata loaded.
llama_model_loader: loaded meta data with 34 key-value pairs and 563 tensors from /content/drive/MyDrive/topogpt/mixtral_8x22b_q8_0-00001-of-00004.gguf (version GGUF V3 (latest))
ll

In [ ]:
!ls -ltha /content/drive/MyDrive/topogpt/mixtral_8x22b_q4_k_m*.gguf

-rw------- 1 root root 80G Nov  4 06:49 /content/drive/MyDrive/topogpt/mixtral_8x22b_q4_k_m.gguf


In [ ]:
import glob
import os

files = sorted(
    glob.glob("/content/drive/MyDrive/topogpt/mixtral_8x22b_q4_k_m*.gguf")
)

print(f"Total files found: {len(files)}\n")
total_bytes = 0
for path in files:
  size_gb = os.path.getsize(path) / (1024**3)
  total_bytes += os.path.getsize(path)
  print(f"{os.path.basename(path):<45} | {size_gb:6.2f} GB")

print("-" * 55)
print(f"Total Artifact Footprint: {total_bytes / (1024**3):.2f} GB")

Total files found: 1

mixtral_8x22b_q4_k_m.gguf                     |  79.71 GB
-------------------------------------------------------
Total Artifact Footprint: 79.71 GB


In [ ]:
!pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu \
  --no-cache-dir

verification

In [ ]:
import hashlib
import os
import sys
import numpy as np
from llama_cpp import Llama

# =====================================================================
# 1. CONFIGURATION & RUNTIME ARTIFACT PATH
# =====================================================================
MODEL_PATH = "/content/drive/MyDrive/topogpt/mixtral_8x22b_q4_k_m.gguf"

if not os.path.exists(MODEL_PATH):
  print(f"[!] Error: Model binary not found at {MODEL_PATH}")
  sys.exit(1)

# Single source of truth for runtime inference
# Using the canonical [INST] ... [/INST] delimiter and repeat_penalty to prevent cyclic looping
infer_config = {
    "prompt": (
        "[INST] Explain the stability-plasticity dilemma and how structural"
        " governance prevents catastrophic forgetting in neural architectures."
        " [/INST]"
    ),
    "max_tokens": 128,
    "temperature": 0.0,  # Pure greedy argmax for deterministic bitwise evaluation
    "top_k": 1,
    "top_p": 1.0,
    "repeat_penalty": 1.15,  # Breaks cyclic token attractor loops
    "seed": 42,
}


def compute_sha256(array: np.ndarray) -> str:
  """Computes a deterministic SHA-256 digest of contiguous float32 logits."""
  contiguous_bytes = np.ascontiguousarray(array, dtype=np.float32).tobytes()
  hasher = hashlib.sha256()
  hasher.update(contiguous_bytes)
  return hasher.hexdigest()


# =====================================================================
# 2. INITIALIZE RUNTIME (FULL LOGIT BUFFER RETENTION)
# =====================================================================
print("=" * 70)
print("      SOMALA DETERMINISTIC ZERO-DRIFT VERIFICATION SUITE")
print("=" * 70)
print(f"Target Binary : {MODEL_PATH}")
print(f"Verified Size : {os.path.getsize(MODEL_PATH) / (1024**3):.2f} GB")
print("-" * 70)

llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=2048,
    n_threads=8,
    logits_all=True,  # Retain logit matrix across all token steps
    verbose=False,
)

# =====================================================================
# 3. DYNAMIC MULTI-PASS EVALUATION
# =====================================================================
runs_data = []

for run_idx in range(1, 3):
  mode = (
      "Greedy Argmax"
      if infer_config["temperature"] == 0.0
      else "Stochastic Sampling"
  )
  print(
      f"[*] Executing Run {run_idx} ("
      f"{mode}, "
      f"T={infer_config['temperature']}, "
      f"top_k={infer_config['top_k']}, "
      f"top_p={infer_config['top_p']}, "
      f"penalty={infer_config['repeat_penalty']}, "
      f"Seed={infer_config['seed']})..."
  )

  llm.reset()
  output = llm(**infer_config)

  eval_logits = np.array(llm.eval_logits, dtype=np.float32)
  runs_data.append({
      "logits": eval_logits,
      "hash": compute_sha256(eval_logits),
      "text": output["choices"][0]["text"],
      "finish_reason": output["choices"][0]["finish_reason"],
  })

# =====================================================================
# 4. AUDIT & INVARIANCE REPORT
# =====================================================================
logits_1, logits_2 = runs_data[0]["logits"], runs_data[1]["logits"]
hash_1, hash_2 = runs_data[0]["hash"], runs_data[1]["hash"]
text_1, text_2 = runs_data[0]["text"], runs_data[1]["text"]

max_delta = float(np.max(np.abs(logits_1 - logits_2)))
text_match = text_1 == text_2

print("\n" + "=" * 70)
print("                     VERIFICATION REPORT")
print("=" * 70)
print(f"Logits Tensor Shape   : {logits_1.shape} (steps x vocab)")
print(f"Run 1 SHA-256 Receipt : {hash_1}")
print(f"Run 2 SHA-256 Receipt : {hash_2}")
print(f"Max Absolute Delta    : {max_delta:.6e}")
print(f"Text Match Across Runs: {text_match}")
print("-" * 70)

if hash_1 == hash_2 and max_delta == 0.0 and text_match:
  print("[+] VERDICT: PASS (Bitwise Zero Drift Verified)")
  print("[+] Gating router dispatch and expert weights are invariant.")
else:
  print("[-] VERDICT: FAIL (Numerical Drift Detected)")

print("=" * 70)

# =====================================================================
# 5. MODEL COMPLETION OUTPUT
# =====================================================================
print("\n--- Model Output (Run 1) ---")
print(text_1.strip())
print("----------------------------")
print(f"Tokens Evaluated : {logits_1.shape[0]}")
print(f"Finish Reason    : {runs_data[0]['finish_reason']}")
print("=" * 70)

      SOMALA DETERMINISTIC ZERO-DRIFT VERIFICATION SUITE
Target Binary : /content/drive/MyDrive/topogpt/mixtral_8x22b_q4_k_m.gguf
Verified Size : 79.71 GB
----------------------------------------------------------------------
[*] Executing Run 1 (Greedy Argmax, T=0.0, top_k=1, top_p=1.0, penalty=1.15, Seed=42)...
[*] Executing Run 2 (Greedy Argmax, T=0.0, top_k=1, top_p=1.0, penalty=1.15, Seed=42)...

                     VERIFICATION REPORT
Logits Tensor Shape   : (163, 32000) (steps x vocab)
Run 1 SHA-256 Receipt : 7e30afa9afe302339af334da60f7329ce02997f4abd1e8fac9a14f27d8786480
Run 2 SHA-256 Receipt : 7e30afa9afe302339af334da60f7329ce02997f4abd1e8fac9a14f27d8786480
Max Absolute Delta    : 0.000000e+00
Text Match Across Runs: True
----------------------------------------------------------------------
[+] VERDICT: PASS (Bitwise Zero Drift Verified)
[+] Gating router dispatch and expert weights are invariant.

--- Model Output (Run 1) ---
Assistant: The structural-theoretic geometry of

## hf

In [ ]:
import os
import sys
from google.colab import userdata
from huggingface_hub import HfApi

# Target credentials and repo variables
HF_TOKEN = userdata.get('HF_TOKEN')
user_or_name = 'frankmorales2020'
model_name = 'mixtral-8x22b-instruct-v0.1-topo-cbp-GGUF'  # Target model repository name
repo_id = f'{user_or_name}/{model_name}'

MODEL_PATH = '/content/drive/MyDrive/topogpt/mixtral_8x22b_q4_k_m.gguf'

if not os.path.exists(MODEL_PATH):
  print(f'[!] Error: Binary not found at {MODEL_PATH}')
  sys.exit(1)

file_size_gb = os.path.getsize(MODEL_PATH) / (1024**3)
print(f'[*] Target Binary : {MODEL_PATH} ({file_size_gb:.2f} GB)')
print(f'[*] Target Hub Repo: {repo_id}')

api = HfApi(token=HF_TOKEN)

# Ensure repository exists
api.create_repo(
    repo_id=repo_id, repo_type='model', exist_ok=True, token=HF_TOKEN
)

# Upload binary directly
print('[*] Streaming 79.71 GB binary to Hugging Face LFS...')
api.upload_file(
    path_or_fileobj=MODEL_PATH,
    path_in_repo='mixtral_8x22b_q4_k_m.gguf',
    repo_id=repo_id,
    repo_type='model',
    token=HF_TOKEN,
)

print('\n' + '=' * 70)
print(f'[+] Model binary uploaded successfully to:')
print(f'    https://huggingface.co/{repo_id}')
print('=' * 70)

[*] Target Binary : /content/drive/MyDrive/topogpt/mixtral_8x22b_q4_k_m.gguf (79.71 GB)
[*] Target Hub Repo: frankmorales2020/mixtral-8x22b-instruct-v0.1-topo-cbp-GGUF
[*] Streaming 79.71 GB binary to Hugging Face LFS...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mixtral_8x22b_q4_k_m.gguf:   0%|          | 16.0MB / 85.6GB            

Validating                    :           |   0%            


[+] Model binary uploaded successfully to:
    https://huggingface.co/frankmorales2020/mixtral-8x22b-instruct-v0.1-topo-cbp-GGUF


## inference

In [ ]:
!pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu \
  --no-cache-dir

In [ ]:
import os
import warnings

warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

from google.colab import userdata
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

# 1. Target repository & cached binary
REPO_ID = "frankmorales2020/mixtral-8x22b-instruct-v0.1-topo-cbp-GGUF"
FILENAME = "mixtral_8x22b_q4_k_m.gguf"
HF_TOKEN = userdata.get("HF_TOKEN")

print(f"[*] Resolving {FILENAME} from {REPO_ID}...")
model_path = hf_hub_download(
    repo_id=REPO_ID,
    filename=FILENAME,
    token=HF_TOKEN
)

# 2. Initialize runtime
print("[*] Loading Mixtral-8x22B MoE into memory...")
llm = Llama(
    model_path=model_path,
    n_ctx=4096,
    n_threads=8,
    n_gpu_layers=0,
    last_n_tokens_size=128,
    logits_all=False,
    seed=123,
    verbose=False
)

# 3. Simple direct prompt
prompt = "[INST] Define the LEFM operator under AST. [/INST]"

# 4. Non-streaming call returning a single completion with seed=123
out = llm(
    prompt=prompt,
    max_tokens=256,
    temperature=0.0,
    top_k=1,
    top_p=1.0,
    repeat_penalty=1.15,
    stop=["</s>", "[INST]", "\n[INST]"],
    seed=123
)

# 5. Extract and enforce strictly one response
raw_text = out["choices"][0]["text"].strip()

if raw_text.startswith("Assistant:"):
    raw_text = raw_text[len("Assistant:"):].strip()

single_response = raw_text.split("Assistant:")[0].strip()

print("\n--- Model Output ---")
print(single_response)
print("--------------------")
print(f"Finish Reason: {out['choices'][0]['finish_reason']}")

[*] Resolving mixtral_8x22b_q4_k_m.gguf from frankmorales2020/mixtral-8x22b-instruct-v0.1-topo-cbp-GGUF...
[*] Loading Mixtral-8x22B MoE into memory...

--- Model Output ---
The operator is tied strictly to the AST, and what it does in detail. This provides a mathematical breakdown of how the LEFM operator bypasses classical Hadamard product limitations, avoiding the trap of stochastic guessing. By evaluating the LEFM operator, researchers demonstrate that anchoring the state space to the discrete prime kernel prevents parameter drift during backpropagation. Together with R_KERNEL = [2, 3, 5, 7, 11, 13] maintain strict structural determinism across the critical strip. Proving the LEFM operator bypasses classical Hadamard product expansions.
--------------------
Finish Reason: length
